# Pesquisa Operacional — Aula 5 · Exercícios Resolvidos
## Teoria das Filas: os 3 exercícios em código

Este notebook resolve, em código, os **3 exercícios** vistos passo a passo nos slides:
1. Identificar os elementos de uma fila (guichê de clínica);
2. Calcular M/M/1 na mão — e conferir com o código (chatbot);
3. Dimensionar M/M/c com Erlang C (o mesmo chatbot, tráfego triplicado).

> **Como usar:** cada caixa cinza é uma *célula* de código. Clique nela e pressione **Shift+Enter**
> para executar, na ordem, de cima para baixo.

In [1]:
# PASSO 0 — Preparar o ambiente.
# Se aparecer erro 'No module named ...', remova o # da linha abaixo, rode, e rode esta célula de novo.
# %pip install numpy

from math import factorial

print('Ambiente OK! Pode seguir para a próxima célula.')

Ambiente OK! Pode seguir para a próxima célula.


## Exercício 1 — Só identificar

> Um guichê único de uma clínica atende pacientes por ordem de chegada. Em média chegam 5 pacientes
> por hora, e cada atendimento leva em média 10 minutos.

Não tem conta de fila aqui ainda — só precisamos **unificar as unidades** antes de identificar os
elementos.

In [2]:
# Exercício 1 — unificando as unidades
lam_min = 5 / 60      # 5 pacientes/hora, em pacientes/minuto (só para conferência)
mu_min = 1 / 10        # 1 atendimento a cada 10 min -> pacientes/minuto

# convertendo tudo para "por hora" (mais natural neste enunciado)
lam_hora = 5           # já está em pacientes/hora
mu_hora = 60 / 10      # 60 min/hora dividido por 10 min/atendimento

print(f"λ = {lam_hora} pacientes/hora")
print(f"μ = {mu_hora} pacientes/hora")
print(f"c = 1 (guichê único)")
print(f"Disciplina = FIFO (\"por ordem de chegada\")")

# Confira: λ e μ estão na MESMA unidade (pacientes/hora)? Essa é a armadilha nº 1 do exercício.

λ = 5 pacientes/hora
μ = 6.0 pacientes/hora
c = 1 (guichê único)
Disciplina = FIFO ("por ordem de chegada")


## Exercício 2 — M/M/1 na mão

> Um chatbot de suporte recebe em média 6 conversas por minuto (λ) e consegue encerrar, em média,
> 8 conversas por minuto (μ). Calcule ρ, L, Lq, W e Wq (em segundos).

A função `mm1` abaixo aplica as mesmas 5 fórmulas que vocês usaram no papel — é para CONFERIR
a conta manual, não para substituí-la.

In [3]:
def mm1(lam, mu):
    """Calcula as métricas do modelo M/M/1 (1 servidor)."""
    rho = lam / mu
    L = rho / (1 - rho)
    Lq = rho**2 / (1 - rho)
    W = L / lam
    Wq = Lq / lam
    return {'rho': rho, 'L': L, 'Lq': Lq, 'W': W, 'Wq': Wq}

r2 = mm1(lam=6, mu=8)
print(f"ρ  : {r2['rho']:.2f}")
print(f"L  : {r2['L']:.2f} conversas no sistema")
print(f"Lq : {r2['Lq']:.2f} conversas na fila")
print(f"W  : {r2['W']*60:.1f} s")
print(f"Wq : {r2['Wq']*60:.1f} s")

# Confira com a mão: ρ=0,75, L=3, Lq=2,25, W=30s, Wq=22,5s

ρ  : 0.75
L  : 3.00 conversas no sistema
Lq : 2.25 conversas na fila
W  : 30.0 s
Wq : 22.5 s


In [4]:
# Conferindo com a Lei de Little
lam = 6
print("L  == λ·W  ?", round(r2['L'], 6), "==", round(lam * r2['W'], 6))
print("Lq == λ·Wq ?", round(r2['Lq'], 6), "==", round(lam * r2['Wq'], 6))

# Se as duas linhas baterem, a conta do Exercício 2 está correta.

L  == λ·W  ? 3.0 == 3.0
Lq == λ·Wq ? 2.25 == 2.25


## Exercício 3 — M/M/c com Erlang C

> Se o tráfego do chatbot triplicasse (λ = 18/min) e cada atendente automatizado desse conta de
> μ = 8/min, quantos "atendentes" em paralelo (c) seriam necessários para manter Wq abaixo de 10
> segundos?

A função `erlang_c` abaixo é a mesma usada na Aula 5 — não precisa reimplementar, só usar.

In [5]:
def erlang_c(lam, mu, c):
    """Calcula Wq para uma fila M/M/c (c servidores idênticos)."""
    a = lam / mu          # carga oferecida, em "Erlangs"
    rho = a / c            # utilização por servidor

    if rho >= 1:
        return {'rho': rho, 'Wq': float('inf'), 'P_espera': 1.0}

    soma = sum(a**k / factorial(k) for k in range(c))
    ultimo_termo = (a**c) / (factorial(c) * (1 - rho))
    P0 = 1 / (soma + ultimo_termo)

    P_espera = ultimo_termo * P0
    Wq = P_espera / (c * mu - lam)

    return {'rho': rho, 'Wq': Wq, 'P_espera': P_espera}

lam3, mu3 = 18, 8
a = lam3 / mu3
print(f"Carga oferecida a = λ/μ = {a:.2f} Erlangs")
print(f"→ c precisa ser pelo menos {int(a) + 1} só para o sistema ser estável (ρ < 1)\n")

limite_wq_s = 10  # segundos

print(f"{'c':<4} {'ρ':>8} {'Wq (s)':>12} {'Dentro do limite?':>20}")
print('-' * 48)
for c in range(1, 6):
    r = erlang_c(lam3, mu3, c)
    wq_s = r['Wq'] * 60 if r['Wq'] != float('inf') else float('inf')
    dentro = 'SIM ✔' if wq_s <= limite_wq_s else 'não'
    print(f"{c:<4} {r['rho']:>8.3f} {wq_s:>12.2f} {dentro:>20}")

# Qual foi o primeiro c "estável"? E o primeiro que ficou dentro do limite de 10s?
# (Resposta esperada: c=1 e c=2 são instáveis; c=3 já é estável E fica dentro do limite.)

Carga oferecida a = λ/μ = 2.25 Erlangs
→ c precisa ser pelo menos 3 só para o sistema ser estável (ρ < 1)

c           ρ       Wq (s)    Dentro do limite?
------------------------------------------------
1       2.250          inf                  não
2       1.125          inf                  não
3       0.750         5.68                SIM ✔
4       0.562         1.03                SIM ✔
5       0.450         0.25                SIM ✔


In [6]:
# Verificação de sanidade: Erlang C com c=1 deve reproduzir o M/M/1 do Exercício 2
check = erlang_c(lam=6, mu=8, c=1)
print(f"Erlang C (c=1): Wq = {check['Wq']*60:.2f} s")
print(f"M/M/1 direto  : Wq = {mm1(lam=6, mu=8)['Wq']*60:.2f} s")
# Os dois devem bater — é a mesma matemática vista de dois jeitos.

Erlang C (c=1): Wq = 22.50 s
M/M/1 direto  : Wq = 22.50 s


## Resumo

✅ Exercício 1: sempre unificar as unidades de λ e μ antes de identificar os elementos da fila;
✅ Exercício 2: as 5 fórmulas do M/M/1 batem com a Lei de Little (L=λW, Lq=λWq);
✅ Exercício 3: a carga oferecida (a=λ/μ) dá o número MÍNIMO de servidores para estabilidade — e o
Erlang C decide, entre os estáveis, qual realmente atende o limite de espera pedido.